In [135]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from snake.snake_gym import Snake

from gymnasium.wrappers import FrameStackObservation, RecordVideo, RecordEpisodeStatistics
from gymnasium.vector import SyncVectorEnv

import numpy as np

import torch
import torch.nn as nn
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from torch.utils.tensorboard import SummaryWriter

import time
import matplotlib.pyplot as plt

In [136]:
model_path = "../models/ppo_cnn_transformer"
prev_version = "v1"
model_version = "v1"

train_episodes = 200
num_steps = 2048
num_envs = 16
stack_size = 4

In [137]:
def make_env(idx: int):
    def thunk():
        env = Snake(truncation_steps=4096)
        
        env = FrameStackObservation(env, stack_size=stack_size)        
        env = RecordEpisodeStatistics(env=env)
    
        return env
    return thunk

In [138]:
envs = SyncVectorEnv([make_env(idx) for idx in range(num_envs)])

In [139]:
action_size = envs.single_action_space.n
action_size

np.int64(4)

In [140]:
envs.single_observation_space

Box(0.0, 3.0, (4, 30, 30), float32)

In [141]:
obs, _ = envs.reset()
obs.shape

(16, 4, 30, 30)

In [142]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [143]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

In [144]:
class AgentCNN(nn.Module):
    def __init__(self, n_hid, n_out):
        super().__init__()
        self.network = nn.Sequential(
            layer_init(nn.Conv2d(stack_size, 32, kernel_size=3, stride=1, padding=1)),
            nn.ReLU(),
            
            layer_init(nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)),
            nn.ReLU(),         
               
            layer_init(nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1)),
            nn.ReLU(),
            
            nn.Flatten(),
            layer_init(nn.Linear(64 * 8 * 8, n_hid)),
            nn.ReLU(),  
        )
        
        self.actor = layer_init(nn.Linear(n_hid, n_out), std=0.01)
        self.critic = layer_init(nn.Linear(n_hid, 1), std=1)                

    def get_value(self, x):
        return self.critic(self.network(x))
    
    def get_action(self, x):
        hidden = self.network(x)
        logits = self.actor(hidden)
        
        action = torch.argmax(logits, dim=1)
        
        return action.item()

    def get_action_and_value(self, x, action=None):
        hidden = self.network(x)
        logits = self.actor(hidden)
        probs = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(hidden)
    
old_agent = AgentCNN(512, action_size)
old_agent.load_state_dict(torch.load("../models/ppo_cnn/agent-v12.pth"))

<All keys matched successfully>

In [145]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.network = nn.Sequential(
            layer_init(nn.Conv2d(stack_size, 32, kernel_size=3, stride=1, padding=1)),
            nn.ReLU(),
            
            layer_init(nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)),
            nn.ReLU(),         
                
            layer_init(nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1)),
            nn.ReLU(),  
        )
        
    def forward(self, x):
        return self.network(x)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head ** -0.5

        self.norm = nn.LayerNorm(dim)

        self.attend = nn.Softmax(dim = -1)
        self.dropout = nn.Dropout(dropout)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()
        
    def forward(self, x):
        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)


class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([])

        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(dim, heads = heads, dim_head = dim_head, dropout = dropout),
                FeedForward(dim, mlp_dim, dropout = dropout)
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x

        return self.norm(x)

class Agent(nn.Module):
    def __init__(self, n_hid, n_out):
        super().__init__()
        
        self.cnn = CNNEncoder()
        
        self.transformer = Transformer(dim=64, depth=2, heads=4, dim_head=16, mlp_dim=128)
        
        self.fc = nn.Sequential(
            layer_init(nn.Linear(64, n_hid)),
            nn.ReLU()
        )
        
        self.pos_embedding = nn.Parameter(torch.randn(1, 64, 64))
        
        self.actor = layer_init(nn.Linear(n_hid, n_out), std=0.01)
        self.critic = layer_init(nn.Linear(n_hid, 1), std=1)                

    def encode(self, x):
        x = self.cnn(x)
        x = rearrange(x, 'b c h w -> b (h w) c')
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.fc(x)
        return x

    def get_value(self, x):
        return self.critic(self.encode(x))
    
    def get_action(self, x):
        hidden = self.encode(x)
        logits = self.actor(hidden)
        
        action = torch.argmax(logits, dim=1)
        
        return action.item()

    def get_action_and_value(self, x, action=None):
        hidden = self.encode(x)
        logits = self.actor(hidden)
        probs = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(hidden)

In [146]:
x = torch.randn(8, 4, 30, 30)

sample_agent = Agent(n_hid=128, n_out=action_size)

action, log_prob, entropy, value = sample_agent.get_action_and_value(x)
print(action.shape)
print(log_prob.shape)
print(entropy.shape)
print(value.shape)

torch.Size([8])
torch.Size([8])
torch.Size([8])
torch.Size([8, 1])


In [147]:
def train(
    envs,
    model,
    optimizer,
    updates,
    update_epochs,
    num_steps,
    target_kl = 0.03,
    mini_batch_size = 64,
    gae_lambda = 0.95,
    clip_coef = 0.2,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    norm_advantages = False,
    clip_vloss = False,
    lr_decay = False,
    gamma = 0.999,
    initial_lrs=None,
    min_lr_ratio = 0.1
):
    
    total_timesteps = updates * num_steps * num_envs
    
    writer = SummaryWriter(f"runs_cnn_transformer/snake-{model_version}")
    
    batch_size = num_steps * num_envs
    
    obs = torch.zeros((num_steps, num_envs) + envs.single_observation_space.shape).to(device)
    actions = torch.zeros((num_steps, num_envs) + envs.single_action_space.shape, dtype=torch.long).to(device)
    logprobs = torch.zeros((num_steps, num_envs)).to(device)
    rewards = torch.zeros((num_steps, num_envs)).to(device)
    dones = torch.zeros((num_steps, num_envs)).to(device)
    values = torch.zeros((num_steps, num_envs)).to(device)
    
    global_step = 0
    next_obs, _ = envs.reset()
    next_obs = torch.Tensor(next_obs).to(device)
    next_done = torch.zeros(num_envs).to(device)
    start_time = time.time()
    
    for episode in range(1, updates + 1):      
        total_reward = 0
        
        if lr_decay and initial_lrs != None:
            frac = max(
                0.0,
                1.0 - global_step / total_timesteps
            )

            decay_factor = min_lr_ratio + frac * (1.0 - min_lr_ratio)

            for group, initial_lr in zip(optimizer.param_groups, initial_lrs):
                group["lr"] = initial_lr * decay_factor
        
        for step in range(0, num_steps):
            global_step += num_envs
            obs[step] = next_obs
            dones[step] = next_done
            
            with torch.no_grad():
                action, logprob, _, value = model.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob
            
            next_obs, reward, terminations, truncations, infos = envs.step(action.cpu().numpy())
            next_done = np.logical_or(terminations, truncations)
            rewards[step] = torch.tensor(reward).to(device).view(-1)
            total_reward += float(reward.mean())
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.tensor(next_done, dtype=torch.float32).to(device)
            
            if "episode" in infos:
                ep = infos["episode"]

                for reward, finished, length in zip(
                    ep["r"],
                    ep["_r"],
                    ep["l"],
                ):
                    if finished:
                        writer.add_scalar("charts/episodic_return", reward, global_step)
                        writer.add_scalar("charts/episodic_length", length, global_step)
                
                                    
        with torch.no_grad():
            next_value = model.get_value(next_obs).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    next_nonterminal = 1.0 - next_done
                    next_values = next_value
                else:
                    next_nonterminal = 1.0 - dones[t + 1]
                    next_values = values[t + 1]
                delta = rewards[t] + gamma * next_values * next_nonterminal - values[t]
                advantages[t] = lastgaelam = delta + gamma * gae_lambda * next_nonterminal * lastgaelam
            returns = advantages + values
         
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advanatages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)
        
        b_inds = np.arange(batch_size)
        clipfracs = []
        for epoch in range(update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, batch_size, mini_batch_size):
                end = start + mini_batch_size
                mb_inds = b_inds[start:end]
                
                _, newlogprob, entropy, newvalue = model.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                
                with torch.no_grad():
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > clip_coef).float().mean().item()]
                    
                mb_advantages = b_advanatages[mb_inds]
                if norm_advantages: 
                    mb_advantages = (mb_advantages-mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
                
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                
                new_value = newvalue.view(-1)

                if clip_vloss:
                    v_loss_unclipped = (
                        new_value - b_returns[mb_inds]
                    ) ** 2

                    v_clipped = (
                        b_values[mb_inds]
                        + torch.clamp(
                            new_value - b_values[mb_inds],
                            -clip_coef,
                            clip_coef,
                        )
                    )

                    v_loss_clipped = (
                        v_clipped - b_returns[mb_inds]
                    ) ** 2

                    v_loss_max = torch.max(
                        v_loss_unclipped,
                        v_loss_clipped
                    )

                    v_loss = 0.5 * v_loss_max.mean()

                else:
                    v_loss = 0.5 * (
                        (new_value - b_returns[mb_inds]) ** 2
                    ).mean()
                    
                entropy_loss = entropy.mean()
                loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef
                
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                
            if target_kl is not None and approx_kl > target_kl:
                break
            
        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        
        # writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)
        writer.add_scalar("charts/mean_rollout_reward", total_reward, global_step) 
        print(f"\rSPS: {int(global_step / (time.time() - start_time))} | episode: {episode} | mean rollout reward: {total_reward} | global_step: {global_step}")
        writer.add_scalar("charts/SPS", int(global_step / (time.time() - start_time)), global_step)
            
    envs.close()
    writer.close()

In [148]:
def freeze(module):
    for param in module.parameters():
        param.requires_grad = False
        
def unfreeze(module):
    for param in module.parameters():
        param.requires_grad = True

In [149]:
agent = Agent(512, action_size).to(device)
agent.cnn.network.load_state_dict(old_agent.network[0:6].state_dict())
# agent.load_state_dict(torch.load(f"{model_path}/agent-{prev_version}.pth"))

optimizer = torch.optim.AdamW([
        {"params": agent.cnn.parameters(), "lr": 5e-6},
        {"params": agent.transformer.parameters(), "lr": 5e-5},
        {"params": agent.fc.parameters(), "lr": 5e-5},
        {"params": agent.actor.parameters(), "lr": 1e-4},
        {"params": agent.critic.parameters(), "lr": 1e-4},
    ],
    eps=1e-5,
    weight_decay=1e-4
)

initial_lrs = [
    group["lr"]
    for group in optimizer.param_groups
]

In [150]:
train(
    envs=envs,
    model=agent,
    optimizer=optimizer,
    updates=train_episodes,
    
    update_epochs=4,
    num_steps=num_steps,
    target_kl=0.03,
    mini_batch_size=512,
    gae_lambda=0.95,
    clip_coef=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    
    norm_advantages=True,
    clip_vloss=True,
    lr_decay=True,
    
    gamma=0.999,
    initial_lrs=initial_lrs,
    min_lr_ratio=0.05
)

SPS: 1669 | episode: 1 | mean rollout reward: 15.3125 | global_step: 32768
SPS: 1193 | episode: 2 | mean rollout reward: 18.75 | global_step: 65536
SPS: 1047 | episode: 3 | mean rollout reward: 22.25 | global_step: 98304
SPS: 957 | episode: 4 | mean rollout reward: 33.375 | global_step: 131072
SPS: 920 | episode: 5 | mean rollout reward: 43.875 | global_step: 163840
SPS: 984 | episode: 6 | mean rollout reward: 56.875 | global_step: 196608
SPS: 1032 | episode: 7 | mean rollout reward: 53.9375 | global_step: 229376
SPS: 1074 | episode: 8 | mean rollout reward: 52.375 | global_step: 262144
SPS: 1074 | episode: 9 | mean rollout reward: 53.125 | global_step: 294912
SPS: 1058 | episode: 10 | mean rollout reward: 49.25 | global_step: 327680
SPS: 1050 | episode: 11 | mean rollout reward: 49.125 | global_step: 360448
SPS: 1073 | episode: 12 | mean rollout reward: 45.6875 | global_step: 393216
SPS: 1098 | episode: 13 | mean rollout reward: 42.6875 | global_step: 425984
SPS: 1089 | episode: 14 | 

In [151]:
torch.save(agent.state_dict(), f"{model_path}/agent-{model_version}.pth")
torch.save(optimizer.state_dict(), f"{model_path}/optimizer-{model_version}.pth")